# Module 25: Observability Distributed Tracing SRE — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/resilience_telemetry.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import resilience_telemetry

classes = [n for n, o in inspect.getmembers(resilience_telemetry, inspect.isclass)
           if o.__module__ == 'resilience_telemetry']
functions = [n for n, o in inspect.getmembers(resilience_telemetry, inspect.isfunction)
             if o.__module__ == 'resilience_telemetry']

print('module   : resilience_telemetry')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(resilience_telemetry, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: W3c traceparent header generation and parsing

This is the module's own `test_w3c_traceparent_header_generation_and_parsing` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import pytest
from resilience_telemetry import (
    CircuitBreaker,
    CircuitBreakerOpenException,
    CircuitBreakerState,
    DistributedTracer,
    W3CTraceContext,
)

trace_id = "4bf92f3577b34da6a3ce929d0e0e4736"
span_id = "00f067aa0ba902b7"
header = W3CTraceContext.format_header(trace_id, span_id, sampled=True)
assert header == f"00-{trace_id}-{span_id}-01"

parsed_trace, parsed_span, sampled = W3CTraceContext.parse_header(header)
assert parsed_trace == trace_id
assert parsed_span == span_id
assert sampled is True

# Invalid header
with pytest.raises(ValueError):
    W3CTraceContext.parse_header("invalid-header-string")

print('PASSED: test_w3c_traceparent_header_generation_and_parsing')

## 3. 🔮 Prediction — commit before you run

A request touches 6 services and one is slow. Predict what a trace shows that 6 separate service dashboards cannot.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_distributed_tracer_parent_child_linking`, which tests exactly this property.


In [ ]:
tracer = DistributedTracer()

root = tracer.start_span("root_op")
root_header = W3CTraceContext.format_header(root.trace_id, root.span_id)

child = tracer.start_span("child_op", parent_traceparent=root_header)
tracer.end_span(child)
tracer.end_span(root)

assert child.trace_id == root.trace_id
assert child.parent_span_id == root.span_id
assert root.parent_span_id is None

spans = tracer.get_trace_spans(root.trace_id)
assert len(spans) == 2

print('PASSED: test_distributed_tracer_parent_child_linking')

## 4. Measure it: Circuit breaker state transitions

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_circuit_breaker_state_transitions` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

cb = CircuitBreaker(failure_threshold=2, recovery_timeout_sec=10.0, half_open_success_threshold=2)

def failing_fn():
    raise RuntimeError("DB Down")

def ok_fn():
    return "SUCCESS"

t = 100.0
# Failure 1: Still CLOSED
with pytest.raises(RuntimeError):
    cb.call(failing_fn, current_time=t)
assert cb.state == CircuitBreakerState.CLOSED

# Failure 2: Hits threshold -> Transitions to OPEN
with pytest.raises(RuntimeError):
    cb.call(failing_fn, current_time=t)
assert cb.state == CircuitBreakerState.OPEN

# Call at t=105s (before 10s timeout): Fast fails with CircuitBreakerOpenException
with pytest.raises(CircuitBreakerOpenException):
    cb.call(ok_fn, current_time=t + 5.0)

# Call at t=111s: Timeout elapsed -> Enters HALF_OPEN and executes ok_fn
res1 = cb.call(ok_fn, current_time=t + 11.0)
assert res1 == "SUCCESS"
assert cb.state == CircuitBreakerState.HALF_OPEN

# Second successful probe heals to CLOSED
res2 = cb.call(ok_fn, current_time=t + 12.0)
assert res2 == "SUCCESS"
assert cb.state == CircuitBreakerState.CLOSED

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_circuit_breaker_state_transitions')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(resilience_telemetry) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. A trace shows causality and latency attribution that per-service dashboards cannot.
2. Context propagation is the whole mechanism - break it and you have unrelated spans.
3. SLOs and error budgets turn reliability into a number you can spend.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
